# Shor's Algorithm from Scratch, Part 4: Modular Exponentiation

In [Part 3](403_shor_scratch_controlled_multiplier.ipynb), we built the controlled modular multiplication $U_a$. This time, we stack them to build **modular exponentiation** $a^j \bmod N$. This corresponds to Section III.D, Eqs. (12)(13)(14) of the paper.

$$
\ket{j}\ket{1} \longrightarrow \ket{j}\ket{a^j \bmod N}
$$

If we view $j$ in binary, $a^j = a^{j_0} \cdot (a^2)^{j_1} \cdot (a^4)^{j_2} \cdots$, so we **just line up multiplications controlled by each bit of $j$**. The $U_a$ from last time serves directly as the building block.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Section III.D

## The climax of this part: update in place

A naive construction runs into trouble. Each multiplication $x \to ax \to a^2x \to \cdots$ **requires a new register**. Multiply $n$ times and you need $n$ registers. That blows the qubit count up to $n^2$.

The paper does not add registers. Instead, it updates the contents of the same register in three stages: **multiply, swap, and multiply back by $a^{-1}$**.

1. Compute $ax \bmod N$ into the result register $b$ (the multiplier from last time)
2. **Swap** $x$ and $b$ — now the $x$ side holds $ax$
3. Run the multiplier by $a^{-1}$ in reverse to return $b$ to $\ket 0$

If $a$ and $N$ are coprime, $a^{-1} \bmod N$ always exists, so the third stage always works.

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## The swap can be "unconditional"

Here is where last time's foreshadowing pays off.

In Eq. (11) last time, when $c=0$ we copied $\ket x$ (not $\ket 0$) into the result register. Because of that, **the swap does not need to be controlled**.

- When $c=1$: swap $\ket{x,\ ax}$ to get $\ket{ax,\ x}$
- When $c=0$: swapping $\ket{x,\ x}$ leaves it as $\ket{x,\ x}$

Nothing happens when $c=0$ **not because the swap is disabled, but because swapping changes nothing**. If we had omitted the copy last time and used $\ket{x, 0}$, the swap would give $\ket{0, x}$ and $x$ would be lost. The copy in Eq. (11) of the paper exists precisely for this reason.

The swap can be written with three CX gates.

In [2]:
from blueqat import Circuit


def carry(c, a, b, c_next):
    q = Circuit(); q.ccx[a, b, c_next]; q.cx[a, b]; q.ccx[c, b, c_next]; return q

def carry_inv(c, a, b, c_next):
    q = Circuit(); q.ccx[c, b, c_next]; q.cx[a, b]; q.ccx[a, b, c_next]; return q

def sum_gate(c, a, b):
    q = Circuit(); q.cx[a, b]; q.cx[c, b]; return q

def add_circuit(c, a, b, n):
    q = Circuit()
    for i in range(n - 1):
        q += carry(c[i], a[i], b[i], c[i + 1])
    q += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    q.cx[a[n - 1], b[n - 1]]
    q += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        q += carry_inv(c[i], a[i], b[i], c[i + 1])
        q += sum_gate(c[i], a[i], b[i])
    return q

def sub_circuit(c, a, b, n):
    r = Circuit(); r.ops = list(reversed(add_circuit(c, a, b, n).ops)); return r

def set_to_zero_if(t, N, val_N, n):
    """t == 0 のとき N レジスタを 0 にする。自分自身が逆演算。"""
    q = Circuit(); q.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            q.cx[t, N[j]]
    q.x[t]; return q

def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N。論文 Fig.4。"""
    q = Circuit()
    q += add_circuit(c, a, b, n)
    q += sub_circuit(c, N, b, n)
    q.cx[b[n], t]
    g = set_to_zero_if(t, N, val_N, n)
    q += g; q += add_circuit(c, N, b, n); q += g
    q += sub_circuit(c, a, b, n)
    q.x[b[n]]; q.cx[b[n], t]; q.x[b[n]]
    q += add_circuit(c, a, b, n)
    return q

In [3]:
def swap_circuit(p, q_reg):
    """⭐ 無条件スワップ。CX 3つで1組を交換する。"""
    s = Circuit()
    for u, v in zip(p, q_reg):
        s.cx[u, v]; s.cx[v, u]; s.cx[u, v]
    return s

## Controlled multiplication $U_a$ — the in-place update form

We rebuild last time's multiplier into three stages (multiply, swap, multiply back). The adder is the same in every stage, so we assemble it once outside the loop and reuse it.

In [4]:
def layout(n):
    c     = list(range(n))
    const = list(range(n, 2 * n))
    b     = list(range(2 * n, 3 * n + 1))
    N     = list(range(3 * n + 1, 4 * n + 1))
    x     = list(range(4 * n + 1, 5 * n + 1))
    return c, const, b, N, x

def build_Ua(mult_a, val_N, n, c, const, b, N, x, ctrl, t, adder, adder_inv):
    """その場で更新する制御乗算 U_a：掛けて・交換して・a^-1 で掛け戻す。"""
    circ = Circuit()
    # A: b += (a*x) mod N ── Fig.5。制御するのは定数のロードだけ
    for i in range(n):
        cv = (mult_a * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (cv >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]
        circ += load; circ += adder; circ += load
    # 式(11)：ctrl == 0 なら x を b へコピー
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]
    # B: ⭐ 無条件スワップ（ctrl=0 でも |x,x> なので交換しても変わらない）
    circ += swap_circuit(x, b[:n])
    # C: b -= (a^-1 * x) mod N ── b を |0> に戻す
    a_inv = pow(mult_a, -1, val_N)
    for i in range(n):
        cv = (a_inv * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (cv >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]
        circ += load; circ += adder_inv; circ += load
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]
    return circ

In [5]:
def run_Ua(mult_a, val_x, val_N, ctrl_value, n=None, quiet=False):
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    c, const, b, N, x = layout(n)
    ctrl, t = 5 * n + 1, 5 * n + 2
    circ = Circuit(5 * n + 3)
    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1: circ.x[x[i]]
        if (val_N >> i) & 1: circ.x[N[i]]
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)
    adder_inv = Circuit(); adder_inv.ops = list(reversed(adder.ops))
    circ += build_Ua(mult_a, val_N, n, c, const, b, N, x, ctrl, t, adder, adder_inv)
    total = circ.n_qubits
    if not quiet:
        print("  [n=%d, qubits=%d, gates=%d]" % (n, total, len(circ.ops)))
    st = circ.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    rd = lambda reg: int("".join(reversed([st[total - 1 - q] for q in reg])), 2)
    return {"x": rd(x), "b": rd(b), "N": rd(N), "const": rd(const),
            "c": rd(c), "t": int(st[total - 1 - t]), "qubits": total,
            "gates": len(circ.ops)}

We verify that $x$ is left unchanged when $c=0$, that multiplication happens only when $c=1$, and that **all work registers return to $\ket 0$**. If they do not return, interference in the superposition breaks down at later stages.

In [6]:
for a, x, ctrl in [(2, 1, 1), (2, 2, 1), (2, 1, 0), (2, 2, 0), (2, 0, 1)]:
    r = run_Ua(a, x, 3, ctrl, 2, quiet=True)
    expected = (a * x) % 3 if ctrl else x
    clean = (r["b"] == 0 and r["N"] == 3 and r["const"] == 0
             and r["c"] == 0 and r["t"] == 0)
    ok = r["x"] == expected and clean
    print(f"ctrl={ctrl}: {a}*{x} mod 3 -> x={r['x']} (expected {expected}), "
          f"scratch clean={clean}  {'OK' if ok else 'MISMATCH'}")
print(f"[{r['qubits']} qubits, {r['gates']} gates]")

ctrl=1: 2*1 mod 3 -> x=2 (expected 2), scratch clean=True  OK


ctrl=1: 2*2 mod 3 -> x=1 (expected 1), scratch clean=True  OK


ctrl=0: 2*1 mod 3 -> x=1 (expected 1), scratch clean=True  OK


ctrl=0: 2*2 mod 3 -> x=2 (expected 2), scratch clean=True  OK


ctrl=1: 2*0 mod 3 -> x=0 (expected 0), scratch clean=True  OK
[13 qubits, 354 gates]


## Modular exponentiation

Now that $U_a$ is a building block, all that remains is to control $U_{a^{2^i}}$ with each bit $j_i$ of the exponent register. The values $a^{2^i} \bmod N$ are precomputed classically (Eq. (13) of the paper).

In [7]:
def build_modexp(mult_a, val_N, exp_val, n, n_exp):
    """|j>|1> → |j>|a^j mod N>。j は古典の値として X で置く。"""
    c, const, b, N, x = layout(n)
    t   = 5 * n + 2
    exp = list(range(5 * n + 3, 5 * n + 3 + n_exp))   # 指数レジスタ
    circ = Circuit(5 * n + 3 + n_exp)
    for i in range(n_exp):
        if (exp_val >> i) & 1:
            circ.x[exp[i]]
    circ.x[x[0]]                                      # x := 1 から始める
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)
    adder_inv = Circuit(); adder_inv.ops = list(reversed(adder.ops))
    for i in range(n_exp):
        a_i = pow(mult_a, 1 << i, val_N)              # a^(2^i) mod N
        circ += build_Ua(a_i, val_N, n, c, const, b, N, x, exp[i], t,
                         adder, adder_inv)
    return circ, x, b, N, c, const, t

def run_modexp(mult_a, val_N, exp_val, n=None, n_exp=None, quiet=False):
    if n is None:
        n = val_N.bit_length()
    if n_exp is None:
        n_exp = max(1, exp_val.bit_length())
    circ, x, b, N, c, const, t = build_modexp(mult_a, val_N, exp_val, n, n_exp)
    total = circ.n_qubits
    if not quiet:
        print("  [n=%d, exp bits=%d, qubits=%d, gates=%d]"
              % (n, n_exp, total, len(circ.ops)))
    st = circ.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    rd = lambda reg: int("".join(reversed([st[total - 1 - q] for q in reg])), 2)
    return {"x": rd(x), "b": rd(b), "N": rd(N), "const": rd(const),
            "c": rd(c), "t": int(st[total - 1 - t]),
            "qubits": total, "gates": len(circ.ops)}

In [8]:
for j in range(4):
    r = run_modexp(2, 3, j, 2, 2, quiet=True)
    expected = pow(2, j, 3)
    clean = r["b"] == 0 and r["const"] == 0 and r["c"] == 0 and r["t"] == 0
    print(f"2^{j} mod 3 = {r['x']}  (Python: {expected}, "
          f"scratch clean={clean})  {'OK' if r['x'] == expected and clean else 'MISMATCH'}")
print(f"[{r['qubits']} qubits, {r['gates']} gates]")

2^0 mod 3 = 1  (Python: 1, scratch clean=True)  OK


2^1 mod 3 = 2  (Python: 2, scratch clean=True)  OK


2^2 mod 3 = 1  (Python: 1, scratch clean=True)  OK


2^3 mod 3 = 2  (Python: 2, scratch clean=True)  OK
[15 qubits, 706 gates]


## Exhaustive test

For $N=3$ and $N=5$, we test every $a$ coprime to $N$ × every $x$ × both values of $c$. Beyond checking that the values are correct, we also check **whether the work registers return to $\ket 0$** every time.

Measured (statevector):

| | Qubits | Cases | Result | Time |
|---|---|---|---|---|
| $N=3$ ($n=2$) | 13 | 12 | All passed | 15 s |
| $N=5$ ($n=3$) | 18 | 40 | All passed | 1293 s |

For $N=7$ and above, the state vector is too heavy, so exhaustive testing is not practical. One case for $N=3$ takes about 1.3 s, whereas $N=5$ takes about 32 s. Five more qubits means 32× the amplitudes, and the measured time was about 24× as well.

In [9]:
from math import gcd

for N in (3,):                      # N=5 は 1293 秒かかるのでここでは N=3 だけ
    n = N.bit_length()
    ok = ng = 0
    for a in range(1, N):
        if gcd(a, N) != 1:
            continue
        for x in range(N):
            for ctrl in (0, 1):
                r = run_Ua(a, x, N, ctrl, n, quiet=True)
                expected = (a * x) % N if ctrl else x
                good = (r["x"] == expected and r["b"] == 0 and r["N"] == N
                        and r["const"] == 0 and r["c"] == 0 and r["t"] == 0)
                ok += good
                ng += not good
    print(f"N={N} (n={n}): PASS {ok} / FAIL {ng}")

N=3 (n=2): PASS 12 / FAIL 0


## Scale

Last time's multiplier used $5n+3$ qubits. Modular exponentiation just adds the exponent register, so it is $5n+3+n_{exp}$. The design of updating in place without adding registers pays off directly.

Measured ($a=2$, $N=3$, $n=2$, 2-bit exponent): **15 qubits / 706 gates**.

Section IV of the paper estimates modular exponentiation at $O(n^3)$ elementary gates. Here we line up $U_a$ $n_{exp}$ times, use the modular adder $2n$ times inside $U_a$, and the modular adder is $O(n)$ gates, giving the same $O(n^3)$.

On the other hand, the cost of classical simulation is **exponential** in the number of qubits. $N=3$ takes 13 qubits and 1.3 s; $N=5$ takes 18 qubits and 32 s. This gap is not because the circuit got larger, but because the state vector grew from $2^{13}$ to $2^{18}$.

## Summary

- Modular exponentiation is **just lining up multiplications controlled by each bit of the exponent** (Eqs. (12)(13))
- To avoid adding registers, we update in place in three stages: **multiply, swap, and multiply back by $a^{-1}$**
- **The swap needs no control.** Because we copied $\ket x$ when $c=0$ in Eq. (11) last time, swapping leaves the state unchanged. That copy was the foreshadowing for this
- 12 cases for $N=3$ and 40 cases for $N=5$ all passed, including value correctness and recovery of the work bits
- 15 qubits / 706 gates ($N=3$, 2-bit exponent)

## Next time

By now, we can compute $a^j \bmod N$ given $j$ as a **classical value**.

In the final part, we place $H$ instead of $X$ in the exponent register. **That alone** computes $a^j \bmod N$ for all $j$ simultaneously. Not a single character of the circuit changes. Apply the inverse quantum Fourier transform to extract the order, and connect it to the classical post-processing from Part 1, and Shor's algorithm is connected from start to finish.